# FireMap v2 — 개선 모델

## v1 대비 변경사항

| 항목 | v1 | v2 |
|------|----|----|  
| Y 변환 | 원값 | **log(Y)** → 역변환 |
| 교차검증 | KFold (무작위) | **GroupKFold (구 단위)** |
| 기본 모델 | RandomForest | **Ridge** (동등 성능, 해석성 ↑) |
| multi_use_cnt | 포함 (93% 제로) | **제거** |
| 화재 파생 피처 | fire_injury, log_fire_damage | **per_fire 정규화** (심각도 지표) |
| 해석 도구 | feature_importances_ | **SHAP values** |
| 베이스라인 | 없음 | **naive + Ridge + RF 비교** |

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.cm as cm
import seaborn as sns
import shap
import pickle, os, warnings
warnings.filterwarnings('ignore')

from sklearn.linear_model import Ridge, Lasso, ElasticNet
from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import GroupKFold, KFold
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.metrics import r2_score, mean_squared_error, mean_absolute_error
import xgboost as xgb

plt.rcParams['font.family'] = 'Malgun Gothic'
plt.rcParams['axes.unicode_minus'] = False
os.makedirs('../models', exist_ok=True)
print('로드 완료')

## 1. 데이터 로드 및 피처 엔지니어링

In [ ]:
df = pd.read_csv('../data/processed/master_table.csv', dtype={'adm_cd': str})

# 결측치: 중앙값 대체
for col in ['avg_living_pop', 'avg_65plus_pop', 'ratio_65plus', 'fire_rate_per_10k']:
    df[col] = df[col].fillna(df[col].median())

df['gu_cd'] = df['adm_cd'].str[:5]

# ── 피처 엔지니어링 ──────────────────────────────────────────
df['log_avg_living_pop'] = np.log1p(df['avg_living_pop'])

# 화재 파생: per_fire 정규화 → 화재당 심각도 (건수가 아닌 강도)
df['damage_per_fire']  = np.log1p(df['fire_damage']  / (df['fire_count'] + 1))
df['injury_per_fire']  = df['fire_injury'] / (df['fire_count'] + 1)
df['death_per_fire']   = df['fire_death']  / (df['fire_count'] + 1)

# ── Y 변환 ───────────────────────────────────────────────────
y_raw = df['fire_rate_per_10k'].copy()
y     = np.log(y_raw)                    # log(Y)

# ── 피처셋 ───────────────────────────────────────────────────
# A: 순수 구조적 (인구·건축 특성만, 화재 데이터 없음)
FEAT_A = [
    'log_avg_living_pop',
    'ratio_65plus',
    'old_bldg_ratio',
    'avg_floors',
]
# B: 구조적 + 피해 심각도 (per_fire 정규화로 건수 누출 차단)
FEAT_B = FEAT_A + [
    'damage_per_fire',
    'injury_per_fire',
]

feat_labels = {
    'log_avg_living_pop': '생활인구 (log)',
    'ratio_65plus':       '65세+ 비율',
    'old_bldg_ratio':     '노후건물비율 (구단위)',
    'avg_floors':         '평균 층수 (구단위)',
    'damage_per_fire':    '화재당 재산피해 (log)',
    'injury_per_fire':    '화재당 부상자 수',
}

print(f'행정동: {len(df)}개  |  구(GroupKFold 그룹): {df["gu_cd"].nunique()}개')
print(f'Y(log) 평균: {y.mean():.3f}  왜도: {y.skew():.3f}  (원값 왜도: {y_raw.skew():.3f})')

## 2. Y 분포 비교 (원값 vs log 변환)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

for ax, vals, title, xlabel in [
    (axes[0], y_raw, f'원값  (왜도={y_raw.skew():.2f})',   '화재율 (1만명당)'),
    (axes[1], y,     f'log변환 (왜도={y.skew():.2f})', 'log(화재율)'),
]:
    ax.hist(vals, bins=30, color='#e74c3c', alpha=0.75, edgecolor='black')
    ax.axvline(vals.mean(), color='navy', linestyle='--', label=f'평균={vals.mean():.2f}')
    ax.set_title(title, fontsize=12)
    ax.set_xlabel(xlabel)
    ax.set_ylabel('빈도')
    ax.legend()

plt.suptitle('Y 변환 전후 분포 비교', fontsize=13, y=1.02)
plt.tight_layout()
plt.savefig('../data/processed/v2_y_transform.png', dpi=150, bbox_inches='tight')
plt.show()

## 3. 모델 비교 (베이스라인 포함)

In [ ]:
groups = df['gu_cd']
gkf = GroupKFold(n_splits=5)   # 구 단위 공간 교차검증
kf  = KFold(n_splits=5, shuffle=True, random_state=42)  # 비교용

def run_cv(model_factory, X, y, cv, groups=None, log_y=True):
    """
    model_factory: callable() → 새 모델 반환
    log_y: True면 log(Y) 예측 후 exp 역변환
    """
    splits = cv.split(X, y, groups=groups) if groups is not None else cv.split(X, y)
    oof = np.zeros(len(X))
    for tr, val in splits:
        m = model_factory()
        m.fit(X.iloc[tr], y.iloc[tr])
        oof[val] = m.predict(X.iloc[val])
    if log_y:
        oof_orig = np.exp(oof)
        y_orig   = np.exp(y)
    else:
        oof_orig, y_orig = oof, y
    return {
        'oof': oof,
        'r2':   r2_score(y_orig, oof_orig),
        'rmse': np.sqrt(mean_squared_error(y_orig, oof_orig)),
        'mae':  mean_absolute_error(y_orig, oof_orig),
        'r2_log': r2_score(y, oof) if log_y else None,
    }

X_A = df[FEAT_A]
X_B = df[FEAT_B]

print('GroupKFold (구 단위 공간 교차검증) 실행 중...')
results = {}

In [ ]:
# ── 베이스라인: 평균 예측 ─────────────────────────────────────
oof_naive = np.full(len(y), y.mean())
naive_r2_orig = r2_score(np.exp(y), np.exp(oof_naive))
results['00_Naive(평균)'] = {
    'r2': naive_r2_orig, 'rmse': np.sqrt(mean_squared_error(np.exp(y), np.exp(oof_naive))),
    'mae': mean_absolute_error(np.exp(y), np.exp(oof_naive)), 'r2_log': 0.0, 'oof': oof_naive
}
print(f"Naive     R²={naive_r2_orig:.4f}")

# ── Ridge (A: 순수 구조적) ────────────────────────────────────
def make_ridge(): return Pipeline([('sc', StandardScaler()), ('m', Ridge(alpha=1.0))])
res = run_cv(make_ridge, X_A, y, gkf, groups=groups)
results['01_Ridge_A(구조적)'] = res
print(f"Ridge-A   R²={res['r2']:.4f}  RMSE={res['rmse']:.4f}  R²_log={res['r2_log']:.4f}")

# ── Ridge (B: 구조적 + 심각도) ───────────────────────────────
res = run_cv(make_ridge, X_B, y, gkf, groups=groups)
results['02_Ridge_B(심각도포함)'] = res
print(f"Ridge-B   R²={res['r2']:.4f}  RMSE={res['rmse']:.4f}  R²_log={res['r2_log']:.4f}")

# ── RF (B, GroupKFold) ────────────────────────────────────────
def make_rf(): return RandomForestRegressor(n_estimators=300, max_depth=6, min_samples_leaf=4, max_features='sqrt', random_state=42, n_jobs=-1)
res = run_cv(make_rf, X_B, y, gkf, groups=groups)
results['03_RF_B(공간CV)'] = res
print(f"RF-B(gkf) R²={res['r2']:.4f}  RMSE={res['rmse']:.4f}  R²_log={res['r2_log']:.4f}")

# ── XGBoost (B, GroupKFold) ───────────────────────────────────
def make_xgb(): return xgb.XGBRegressor(n_estimators=300, max_depth=4, learning_rate=0.05, subsample=0.8, colsample_bytree=0.8, reg_alpha=0.1, reg_lambda=1.0, random_state=42, verbosity=0, n_jobs=-1)
res = run_cv(make_xgb, X_B, y, gkf, groups=groups)
results['04_XGB_B(공간CV)'] = res
print(f"XGB-B(gkf) R²={res['r2']:.4f}  RMSE={res['rmse']:.4f}  R²_log={res['r2_log']:.4f}")

# ── 참고: v1 방식 (random KFold) ─────────────────────────────
res_v1 = run_cv(make_rf, X_B, y_raw, kf, groups=None, log_y=False)
print(f"\n[참고] v1 RF random KFold (원값Y): R²={res_v1['r2']:.4f}")

In [ ]:
# 결과 비교 테이블
cmp = pd.DataFrame([
    {'모델': k, 'R²(원값)': v['r2'], 'RMSE': v['rmse'], 'MAE': v['mae']}
    for k, v in results.items()
]).round(4)
print('=== 모델 비교 (GroupKFold, log(Y) 역변환 기준) ===')
print(cmp.to_string(index=False))

In [ ]:
# 시각화
fig, axes = plt.subplots(1, 2, figsize=(13, 5))

names  = [k.split('_', 1)[1] for k in results.keys()]
r2s    = [v['r2']   for v in results.values()]
rmses  = [v['rmse'] for v in results.values()]
colors = ['#95a5a6', '#3498db', '#e67e22', '#e74c3c', '#8e44ad']

for ax, vals, title, ylabel in [
    (axes[0], r2s,   'R² 비교 (높을수록 좋음)',  'R²'),
    (axes[1], rmses, 'RMSE 비교 (낮을수록 좋음)', 'RMSE (화재율)'),
]:
    bars = ax.bar(names, vals, color=colors[:len(names)], alpha=0.85, edgecolor='black')
    for bar, v in zip(bars, vals):
        ax.text(bar.get_x() + bar.get_width()/2, v + (max(vals)*0.01),
                f'{v:.3f}', ha='center', fontsize=9, fontweight='bold')
    ax.set_title(title, fontsize=12)
    ax.set_ylabel(ylabel)
    ax.tick_params(axis='x', rotation=30)

plt.suptitle('v2 모델 비교 (GroupKFold, 구 단위 공간 교차검증)', fontsize=13, y=1.02)
plt.tight_layout()
plt.savefig('../data/processed/v2_model_comparison.png', dpi=150, bbox_inches='tight')
plt.show()

## 4. 최종 모델 선택 및 전체 데이터 학습

Ridge-B를 최종 모델로 선택:
- RF와 동등 성능, 계수로 직접 해석 가능
- 공간 교차검증 기준의 정직한 R²

In [ ]:
# Lasso로 최적 alpha 탐색 (피처 선택 효과)
from sklearn.linear_model import LassoCV, RidgeCV

sc = StandardScaler()
X_B_scaled = sc.fit_transform(X_B)

ridge_cv = RidgeCV(alphas=np.logspace(-2, 3, 50), cv=5)
ridge_cv.fit(X_B_scaled, y)
print(f'RidgeCV 최적 alpha: {ridge_cv.alpha_:.4f}')

# 최종 모델
final_model = Pipeline([
    ('sc', StandardScaler()),
    ('m',  Ridge(alpha=ridge_cv.alpha_))
])
final_model.fit(X_B, y)

# 계수 시각화
coef = pd.Series(
    final_model.named_steps['m'].coef_,
    index=[feat_labels.get(f, f) for f in FEAT_B]
).sort_values()

fig, ax = plt.subplots(figsize=(8, 5))
colors_coef = ['#e74c3c' if v > 0 else '#3498db' for v in coef]
coef.plot(kind='barh', ax=ax, color=colors_coef, edgecolor='gray', alpha=0.85)
ax.axvline(0, color='black', linewidth=1)
ax.set_title('Ridge 회귀 계수 (표준화된 값)', fontsize=13)
ax.set_xlabel('계수 (양수=위험도 ↑, 음수=위험도 ↓)')
for i, v in enumerate(coef):
    ax.text(v + (0.002 if v >= 0 else -0.002), i, f'{v:.3f}',
            va='center', ha='left' if v >= 0 else 'right', fontsize=9)
plt.tight_layout()
plt.savefig('../data/processed/v2_ridge_coef.png', dpi=150, bbox_inches='tight')
plt.show()
print('\n계수 해석:')
print(coef.round(4))

## 5. SHAP 분석

In [ ]:
# SHAP: LinearExplainer (Ridge용, 정확한 SHAP값)
sc_fitted   = final_model.named_steps['sc']
ridge_model = final_model.named_steps['m']
X_B_sc      = sc_fitted.transform(X_B)

explainer   = shap.LinearExplainer(ridge_model, X_B_sc)
shap_values = explainer.shap_values(X_B_sc)
shap_df     = pd.DataFrame(shap_values, columns=[feat_labels.get(f, f) for f in FEAT_B])

print(f'SHAP values shape: {shap_df.shape}')
print(f'|SHAP| 평균 (전체 중요도):')
print(shap_df.abs().mean().sort_values(ascending=False).round(4))

In [ ]:
# SHAP 요약 플롯 (beeswarm)
shap.summary_plot(
    shap_values,
    X_B_sc,
    feature_names=[feat_labels.get(f, f) for f in FEAT_B],
    show=False,
    plot_size=(10, 5)
)
plt.title('SHAP Summary (beeswarm) — Ridge 모델', fontsize=13)
plt.tight_layout()
plt.savefig('../data/processed/v2_shap_summary.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# SHAP bar plot (평균 |SHAP|)
shap.summary_plot(
    shap_values,
    X_B_sc,
    feature_names=[feat_labels.get(f, f) for f in FEAT_B],
    plot_type='bar',
    show=False,
    plot_size=(8, 4)
)
plt.title('SHAP 피처 중요도 (평균 |SHAP|)', fontsize=13)
plt.tight_layout()
plt.savefig('../data/processed/v2_shap_bar.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# SHAP waterfall — 고위험 행정동 개별 설명
pred_log = final_model.predict(X_B)
top_idx  = np.argmax(pred_log)  # 가장 고위험 행정동

print(f'고위험 행정동: {df.iloc[top_idx]["adm_nm"]}')
print(f'예측 화재율: {np.exp(pred_log[top_idx]):.2f}  (평균: {y_raw.mean():.2f})')

exp_obj = shap.Explanation(
    values=shap_values[top_idx],
    base_values=explainer.expected_value,
    data=X_B_sc[top_idx],
    feature_names=[feat_labels.get(f, f) for f in FEAT_B]
)
shap.waterfall_plot(exp_obj, show=False, max_display=10)
plt.title(f'SHAP Waterfall — {df.iloc[top_idx]["adm_nm"]}', fontsize=12)
plt.tight_layout()
plt.savefig('../data/processed/v2_shap_waterfall_top.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# SHAP force plot 저장 (HTML)
force_html = shap.force_plot(
    explainer.expected_value,
    shap_values[:20],
    X_B_sc[:20],
    feature_names=[feat_labels.get(f, f) for f in FEAT_B],
    show=False
)
shap.save_html('../data/processed/v2_shap_force.html', force_html)
print('SHAP force plot 저장: data/processed/v2_shap_force.html')

## 6. 예측 결과 및 잔차 분석

In [ ]:
# OOF 예측 (Ridge-B, GroupKFold)
oof_b = results['02_Ridge_B(심각도포함)']['oof']
y_pred_orig = np.exp(oof_b)

fig, axes = plt.subplots(1, 3, figsize=(16, 5))

# 예측 vs 실제
axes[0].scatter(y_raw, y_pred_orig, alpha=0.5, s=20, color='#e74c3c')
lim = (y_raw.min()-1, y_raw.max()+2)
axes[0].plot(lim, lim, 'b--', lw=1.5, label='완전 예측선')
r2_val = results['02_Ridge_B(심각도포함)']['r2']
axes[0].set_title(f'예측 vs 실제 (R²={r2_val:.3f})', fontsize=12)
axes[0].set_xlabel('실제 화재율'); axes[0].set_ylabel('예측 화재율')
axes[0].legend()

# 잔차 (log 스케일)
res_log = y.values - oof_b
axes[1].scatter(oof_b, res_log, alpha=0.5, s=20, color='#3498db')
axes[1].axhline(0, color='black', lw=1)
axes[1].set_xlabel('예측 log(화재율)'); axes[1].set_ylabel('잔차')
axes[1].set_title('잔차 플롯 (log 스케일)', fontsize=12)

# 잔차 분포
axes[2].hist(res_log, bins=25, color='#2ecc71', edgecolor='black', alpha=0.75)
axes[2].set_title(f'잔차 분포  (왜도={pd.Series(res_log).skew():.2f})', fontsize=12)
axes[2].set_xlabel('잔차')

plt.tight_layout()
plt.savefig('../data/processed/v2_residuals.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# 큰 잔차 행정동 (과대평가 / 과소평가)
df['oof_pred']  = y_pred_orig
df['residual']  = y_raw - y_pred_orig
df['abs_resid'] = df['residual'].abs()

print('=== 과대평가 (실제 < 예측) — 예측보다 안전한 지역 ===')
print(df.nsmallest(8, 'residual')[['adm_nm', 'fire_rate_per_10k', 'oof_pred', 'residual']].to_string(index=False))
print()
print('=== 과소평가 (실제 > 예측) — 예측보다 위험한 지역 ===')
print(df.nlargest(8, 'residual')[['adm_nm', 'fire_rate_per_10k', 'oof_pred', 'residual']].to_string(index=False))

## 7. 위험도 스코어 (v2)

In [ ]:
# 전체 데이터 예측 (최종 모델)
pred_log_final = final_model.predict(X_B)
pred_orig      = np.exp(pred_log_final)

p_min, p_max = pred_orig.min(), pred_orig.max()
df['v2_pred_fire_rate'] = pred_orig.round(4)
df['v2_risk_score']     = ((pred_orig - p_min) / (p_max - p_min) * 9 + 1).round(2)

# 5분위 등급 (균등 배분)
q = np.percentile(df['v2_risk_score'], [20, 40, 60, 80])
df['v2_risk_grade'] = pd.cut(
    df['v2_risk_score'],
    bins=[0, q[0], q[1], q[2], q[3], 10.1],
    labels=['매우낮음', '낮음', '보통', '높음', '매우높음'],
    include_lowest=True
)

print('위험 등급 분포:')
print(df['v2_risk_grade'].value_counts().sort_index())

In [ ]:
# v1 vs v2 스코어 비교
try:
    v1_df = pd.read_csv('../data/processed/risk_scores.csv', dtype={'adm_cd': str})
    merged = df[['adm_cd', 'adm_nm', 'v2_risk_score']].merge(
        v1_df[['adm_cd', 'risk_score']].rename(columns={'risk_score': 'v1_risk_score'}),
        on='adm_cd'
    )
    r_corr = merged[['v1_risk_score', 'v2_risk_score']].corr().iloc[0, 1]

    fig, ax = plt.subplots(figsize=(7, 6))
    ax.scatter(merged['v1_risk_score'], merged['v2_risk_score'], alpha=0.5, s=20, color='#e74c3c')
    lim2 = (0, 11)
    ax.plot(lim2, lim2, 'b--', lw=1.2)
    ax.set_xlabel('v1 위험도 스코어')
    ax.set_ylabel('v2 위험도 스코어')
    ax.set_title(f'v1 vs v2 스코어 비교 (r={r_corr:.3f})', fontsize=12)
    plt.tight_layout()
    plt.savefig('../data/processed/v2_vs_v1_score.png', dpi=150, bbox_inches='tight')
    plt.show()
    print(f'v1/v2 상관: {r_corr:.3f}')

    # 순위 변동이 큰 행정동
    merged['v1_rank'] = merged['v1_risk_score'].rank(ascending=False)
    merged['v2_rank'] = merged['v2_risk_score'].rank(ascending=False)
    merged['rank_diff'] = (merged['v1_rank'] - merged['v2_rank']).abs()
    print('\n순위 변동 상위 10 행정동:')
    print(merged.nlargest(10, 'rank_diff')[['adm_nm','v1_rank','v2_rank','rank_diff']].to_string(index=False))
except FileNotFoundError:
    print('v1 risk_scores.csv 없음')

In [ ]:
# 상위 25 고위험 시각화
top25 = df.sort_values('v2_risk_score', ascending=False).head(25)

fig, ax = plt.subplots(figsize=(10, 9))
norm = plt.Normalize(df['v2_risk_score'].min(), df['v2_risk_score'].max())
cmap = plt.cm.RdYlGn_r
colors_bar = [cmap(norm(s)) for s in top25['v2_risk_score'][::-1]]

bars = ax.barh(top25['adm_nm'][::-1], top25['v2_risk_score'][::-1],
               color=colors_bar, edgecolor='gray', linewidth=0.5)
ax.set_xlabel('v2 위험도 스코어 (1-10)', fontsize=12)
ax.set_title('서울시 화재 위험 상위 25 행정동 (v2 모델)', fontsize=14)
ax.set_xlim(0, 11)
for bar, score in zip(bars, top25['v2_risk_score'][::-1]):
    ax.text(score + 0.1, bar.get_y() + bar.get_height()/2,
            f'{score:.1f}', va='center', fontsize=9)

sm = plt.cm.ScalarMappable(cmap=cmap, norm=norm)
sm.set_array([])
plt.colorbar(sm, ax=ax, label='위험도', shrink=0.6)
plt.tight_layout()
plt.savefig('../data/processed/v2_top25_risk.png', dpi=150, bbox_inches='tight')
plt.show()

## 8. 결과 저장

In [ ]:
# SHAP 값을 df에 추가
shap_cols = {f'shap_{feat_labels.get(f, f)}': shap_values[:, i] for i, f in enumerate(FEAT_B)}
shap_contrib = pd.DataFrame(shap_cols, index=df.index)

output_cols = [
    'adm_cd', 'adm_nm', 'gu_cd',
    'fire_rate_per_10k', 'v2_pred_fire_rate', 'v2_risk_score', 'v2_risk_grade',
    'log_avg_living_pop', 'ratio_65plus', 'old_bldg_ratio', 'avg_floors',
    'damage_per_fire', 'injury_per_fire',
    'avg_living_pop', 'fire_count', 'fire_damage', 'fire_injury',
]
result_df = pd.concat([df[output_cols], shap_contrib], axis=1)\
              .sort_values('v2_risk_score', ascending=False)

result_df.to_csv('../data/processed/v2_risk_scores.csv', index=False, encoding='utf-8-sig')
print(f'저장: data/processed/v2_risk_scores.csv  ({len(result_df)}행)')

# 모델 저장
with open('../models/ridge_fire_risk_v2.pkl', 'wb') as f:
    pickle.dump({
        'model':       final_model,
        'features':    FEAT_B,
        'feat_labels': feat_labels,
        'cv_r2':       results['02_Ridge_B(심각도포함)']['r2'],
        'cv_rmse':     results['02_Ridge_B(심각도포함)']['rmse'],
        'shap_values': shap_values,
        'explainer':   explainer,
    }, f)
print('저장: models/ridge_fire_risk_v2.pkl')

In [ ]:
# 최종 요약
print('=' * 55)
print('       FireMap v2 — 최종 결과 요약')
print('=' * 55)
print(f'학습 데이터   : 서울 {len(df)}개 행정동 (2021-2023)')
print(f'교차검증      : GroupKFold (구 단위, 5-fold)')
print(f'Y 변환        : log(fire_rate_per_10k)')
print(f'최종 모델     : Ridge (alpha={ridge_cv.alpha_:.4f})')
print(f'피처 수       : {len(FEAT_B)}개')
print()
for k, v in results.items():
    print(f'  {k:<28} R²={v["r2"]:.4f}  RMSE={v["rmse"]:.4f}')
print()
print('고위험 상위 10 (v2):')
for _, row in df.nlargest(10, 'v2_risk_score')[['adm_nm', 'v2_risk_score']].iterrows():
    print(f'  {row["adm_nm"]:20}  {row["v2_risk_score"]:.1f}')